## Dynamic BETA_ALPHA System

In [1]:
ticker  =  'MSFT'
language = "English"

## Calling Sector_Analyst_Function To Refresh the Relative Asset + Sector Index

In [2]:
import asyncio
import sys
import os
from pathlib import Path

# Get the absolute path to the project root
project_root = Path.cwd().resolve()
print(f"Project root: {project_root}")

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Also add the parent directory in case we're in a subdirectory
parent_dir = project_root.parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

print(f"Python path: {sys.path[:3]}")

# Check if shared_clients.py exists
shared_clients_path = project_root / "shared_clients.py"
print(f"Looking for shared_clients.py at: {shared_clients_path}")
print(f"File exists: {shared_clients_path.exists()}")

# Import shared clients using direct path
import importlib.util
import importlib

try:
    # Method 1: Direct import from absolute path
    spec = importlib.util.spec_from_file_location("shared_clients", shared_clients_path)
    shared_clients_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(shared_clients_module)
    
    # Make it available in the namespace
    import sys
    sys.modules["shared_clients"] = shared_clients_module
    shared_clients = shared_clients_module.shared_clients
    
    print("✅ Successfully imported shared_clients using direct path method")
    
except Exception as e:
    print(f"❌ Direct path import failed: {e}")
    
    # Method 2: Try importing from current directory
    try:
        os.chdir(project_root)
        import shared_clients
        importlib.reload(shared_clients)
        from shared_clients import shared_clients
        print("✅ Successfully imported shared_clients using directory change method")
    except Exception as e2:
        print(f"❌ Directory change import failed: {e2}")
        raise ImportError("Could not import shared_clients using any method")

# Initialize shared clients
print("🚀 Initializing shared clients...")
await shared_clients.initialize()
print("✅ Shared clients initialized")

# Import Sector Analyst Agent
try:
    from Sector_Analyst_Agent import SectorAnalystAgent
    print("✅ Successfully imported SectorAnalystAgent")
except ImportError as e:
    print(f"❌ Failed to import SectorAnalystAgent: {e}")
    # Try alternative import
    sys.path.insert(0, str(project_root))
    from Sector_Analyst_Agent import SectorAnalystAgent
    print("✅ Successfully imported SectorAnalystAgent (alternative method)")

# Import the enhanced Sector Analyst Storage Agent for direct database access
try:
    from Sector_Analyst_Agent.Sector_Analyst_Storage import SectorAnalystStorage
    print("✅ Successfully imported SectorAnalystStorage")
except ImportError as e:
    print(f"❌ Failed to import SectorAnalystStorage: {e}")
    raise ImportError("Could not import SectorAnalystStorage")

async def get_sector_analysis(ticker: str, user_query: str = None, user_id: str = "default_user") -> dict:
    """
    Get sector analysis for a ticker using the Sector Analyst Agent + direct database access for sector index
    
    Args:
        ticker (str): Stock ticker symbol (e.g., 'AAPL')
        user_query (str, optional): Specific user question about the sector analysis
        user_id (str): User ID for database storage (default: "default_user")
    
    Returns:
        dict: Sector analysis results containing:
        - asset_relative: What the company is relative to
        - relative_sector_index: Matched sector index with confidence score and reasoning
        - answer_collection: Sector trend and competitor answers
        - url_collection: URLs for further research
        - status: "success" or "failed"
    """
    try:
        print(f"�� Getting sector analysis for {ticker}...")
        
        # Step 1: Use the existing Sector Analyst Agent for updates/downloads
        agent = SectorAnalystAgent(
            shared_clients=shared_clients,
            user_id=user_id
        )
        
        # Process sector analysis (this handles updates/downloads)
        result = await agent.process_sector_analysis(ticker, user_query)
        
        # Close the agent
        await agent.close()
        
        # Step 2: Direct database access to get the relative_sector_index
        print(f"�� Getting sector index from database for {ticker}...")
        storage_agent = SectorAnalystStorage(shared_clients=shared_clients)
        db_data = await storage_agent.get_sector_data(ticker)
        
        # Step 3: Merge the results
        if result.get("status") == "success":
            # Add the sector index from direct database access
            if db_data and db_data.get('relative_sector_index'):
                result["relative_sector_index"] = db_data.get('relative_sector_index', {})
                print(f"✅ Sector index found: {result['relative_sector_index'].get('ticker', 'N/A')}")
            else:
                result["relative_sector_index"] = {}
                print(f"⚠️ No sector index found in database for {ticker}")
            
            print(f"✅ Sector analysis completed for {ticker}")
            print(f" - Asset relative: {result.get('asset_relative', 'N/A')}")
            print(f" - Sector index: {result.get('relative_sector_index', {}).get('ticker', 'N/A')}")
            print(f" - Answer collection keys: {list(result.get('answer_collection', {}).keys())}")
            print(f" - URL collection keys: {list(result.get('url_collection', {}).keys())}")
        else:
            print(f"❌ Sector analysis failed for {ticker}: {result.get('error', 'Unknown error')}")
        
        return result
        
    except Exception as e:
        print(f"❌ Error getting sector analysis for {ticker}: {e}")
        return {
            "ticker": ticker,
            "asset_relative": "",
            "relative_sector_index": {},
            "answer_collection": {},
            "url_collection": {},
            "error": str(e),
            "status": "failed"
        }





Project root: /Users/xikinki/Desktop/QandQ_AI/Fintegrate_AI_File/New_Fintegrate_AI(V3)/Quant_Impact_Agent
Python path: ['/Users/xikinki/Desktop/QandQ_AI/Fintegrate_AI_File/New_Fintegrate_AI(V3)', '/Users/xikinki/Desktop/QandQ_AI/Fintegrate_AI_File/New_Fintegrate_AI(V3)/Quant_Impact_Agent', '/Users/xikinki/anaconda3/envs/arviz_env/lib/python310.zip']
Looking for shared_clients.py at: /Users/xikinki/Desktop/QandQ_AI/Fintegrate_AI_File/New_Fintegrate_AI(V3)/Quant_Impact_Agent/shared_clients.py
File exists: False
❌ Direct path import failed: [Errno 2] No such file or directory: '/Users/xikinki/Desktop/QandQ_AI/Fintegrate_AI_File/New_Fintegrate_AI(V3)/Quant_Impact_Agent/shared_clients.py'
✅ Successfully imported shared_clients using directory change method
🚀 Initializing shared clients...
🚀 Initializing shared client pool...
🤖 Initializing LLM clients...
🗄️ Initializing Redis pools...
🌐 Initializing HTTP session...
🧪 Testing connections...
✅ All Redis connections verified
⏱️ Initialization 

In [7]:
result = await get_sector_analysis(ticker)
sector_index = result['relative_sector_index']['ticker']
sector_index 

�� Getting sector analysis for MSFT...
🔍 Loading sector index CSV from: /Users/xikinki/Desktop/QandQ_AI/Fintegrate_AI_File/New_Fintegrate_AI(V3)/Sector_Analyst_Agent/sector_index_tickers.csv
✅ CSV file exists, loading...
   Loaded 93 rows, columns: ['Ticker', 'Description']
✅ Loaded 93 sector indices from CSV
   Sample tickers: ['^GSPC', '^DJI', '^IXIC', 'SPY', 'QQQ']
�� Getting sector index from database for MSFT...
🔍 Loading sector index CSV from: /Users/xikinki/Desktop/QandQ_AI/Fintegrate_AI_File/New_Fintegrate_AI(V3)/Sector_Analyst_Agent/sector_index_tickers.csv
✅ CSV file exists, loading...
   Loaded 93 rows, columns: ['Ticker', 'Description']
✅ Loaded 93 sector indices from CSV
   Sample tickers: ['^GSPC', '^DJI', '^IXIC', 'SPY', 'QQQ']
✅ Sector index found: XLK
✅ Sector analysis completed for MSFT
 - Asset relative: Microsoft Corporation operates through three primary business segments: Productivity and Business Processes (including Office Commercial, Office Consumer, LinkedIn, 

'XLK'

## Get Sector Beta

## Double Beta System:
### (R_Sector - Risk_Free) =  Beta_Sector * (Macro - Risk_Free) + Sector_Alpha + Sector_Error 
### (R_Stock - Risk_Free )=  Beta_1 * (Macro - Risk_Free) + Beta_2 * Sector_Error + Alpha 

In [8]:
import numpy as np
import pandas as pd
from scipy import stats
import requests
from datetime import datetime, timedelta

# Your FMP API key (replace with your actual key)
FMP_API_KEY = "9dfbbfa29d93f4793f246e8fb5ca5e74" # Replace with your actual API key

def get_stock_prices_fmp(ticker: str, start_date: str, end_date: str, api_key: str = None) -> pd.DataFrame:
    """Get stock prices from FMP API"""
    if api_key is None:
        api_key = FMP_API_KEY
    
    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{ticker}"
    params = {
        'from': start_date,
        'to': end_date,
        'apikey': api_key
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if 'historical' in data:
            df = pd.DataFrame(data['historical'])
            df['date'] = pd.to_datetime(df['date'])
            df = df.sort_values('date').reset_index(drop=True)
            return df[['date', 'close']]
        else:
            print(f"❌ No historical data found for {ticker}")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"❌ Error fetching data for {ticker}: {e}")
        return pd.DataFrame()

def orthogonalize_sector_analysis_three_params(stock_ticker: str, sector_index: str, market_ticker: str = "SPY", 
                                              period_days: int = 252, risk_free_rate: float = 0.025, api_key: str = None):
    """
    Perform orthogonal sector analysis with risk-free rate adjustment - simplified to 3 parameters
    
    Args:
        stock_ticker: Stock symbol (e.g., 'AAPL')
        sector_index: Sector index symbol (e.g., 'XLK')
        market_ticker: Market benchmark (default: 'SPY')
        period_days: Number of trading days to analyze (default: 252 = 1 year)
        risk_free_rate: Annual risk-free rate (default: 2.5%)
        api_key: FMP API key
    
    Returns:
        dict: Three key parameters (alpha, market_beta, sector_beta)
    """
    
    print(f"�� Orthogonal Sector Analysis: {stock_ticker} vs {market_ticker} + {sector_index}")
    print(f"   Risk-free rate: {risk_free_rate:.1%} annual")
    
    # Calculate date range
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=period_days + 50)).strftime('%Y-%m-%d')
    
    # Convert annual risk-free rate to daily
    risk_free_daily = risk_free_rate / 252
    
    # Get price data
    stock_df = get_stock_prices_fmp(stock_ticker, start_date, end_date, api_key)
    sector_df = get_stock_prices_fmp(sector_index, start_date, end_date, api_key)
    market_df = get_stock_prices_fmp(market_ticker, start_date, end_date, api_key)
    
    if stock_df.empty or sector_df.empty or market_df.empty:
        return {
            "alpha": None,
            "market_beta": None,
            "sector_beta": None,
            "error": "Failed to fetch price data"
        }
    
    # Merge dataframes
    merged_df = stock_df.merge(sector_df, on='date', suffixes=('_stock', '_sector'))
    merged_df = merged_df.merge(market_df, on='date')
    merged_df = merged_df.rename(columns={'close': 'close_market'})
    
    merged_df = merged_df.sort_values('date').reset_index(drop=True)
    merged_df = merged_df.tail(period_days).reset_index(drop=True)
    
    # Calculate daily returns
    merged_df['return_stock'] = merged_df['close_stock'].pct_change()
    merged_df['return_sector'] = merged_df['close_sector'].pct_change()
    merged_df['return_market'] = merged_df['close_market'].pct_change()
    
    # Remove NaN values
    merged_df = merged_df.dropna()
    
    # Calculate excess returns (subtract risk-free rate)
    merged_df['excess_return_stock'] = merged_df['return_stock'] - risk_free_daily
    merged_df['excess_return_sector'] = merged_df['return_sector'] - risk_free_daily
    merged_df['excess_return_market'] = merged_df['return_market'] - risk_free_daily
    
    # First regression - Sector on Market (Orthogonalization) using excess returns
    slope_sector_market, intercept_sector_market, r_value_sector_market, p_value_sector_market, std_err_sector_market = stats.linregress(
        merged_df['excess_return_market'], merged_df['excess_return_sector']
    )
    
    # Calculate sector residuals (pure sector move beyond macro)
    merged_df['sector_residual'] = merged_df['excess_return_sector'] - (slope_sector_market * merged_df['excess_return_market'] + intercept_sector_market)
    
    # Check data sufficiency
    if len(merged_df) < 10:
        print(f"   ❌ Insufficient data ({len(merged_df)} days)")
        return {
            "alpha": None,
            "market_beta": None,
            "sector_beta": None,
            "error": "Insufficient data"
        }
    
    # Prepare data for multiple regression using excess returns
    X = np.column_stack([
        merged_df['excess_return_market'],           # Market excess returns
        merged_df['sector_residual']                 # Orthogonalized sector residuals
    ])
    y = merged_df['excess_return_stock']             # Stock excess returns
    
    # Perform multiple regression: R_stock - Rf = α + β_macro * (R_market - Rf) + β_micro * ε_sector
    X_with_intercept = np.column_stack([np.ones(len(X)), X])
    beta_coeffs = np.linalg.lstsq(X_with_intercept, y, rcond=None)[0]
    
    alpha = beta_coeffs[0] * 252        # Annualized alpha (risk-adjusted)
    market_beta = beta_coeffs[1]        # Beta to market
    sector_beta = beta_coeffs[2]        # Beta to orthogonalized sector
    
    # Calculate R-squared
    y_pred = X_with_intercept @ beta_coeffs
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    print(f"   ✅ Results: α={alpha:.4f}, β_market={market_beta:.4f}, β_sector={sector_beta:.4f}, R²={r_squared:.4f}")
    
    # Return the three key parameters
    three_params = {
        "alpha": alpha,
        "market_beta": market_beta,
        "sector_beta": sector_beta,
        "r_squared": r_squared,
        "risk_free_rate": risk_free_rate,
        "data_points": len(merged_df)
    }
    
    return three_params

def single_beta_test(stock_ticker: str, market_ticker: str = "SPY", 
                    period_days: int = 252, risk_free_rate: float = 0.025, api_key: str = None):
    """
    Simple single beta test with risk-free rate - returns alpha and beta for uptrend/downtrend
    
    Returns:
        dict: Single beta results (alpha and beta only)
    """
    
    print(f"🔍 Single Beta Test: {stock_ticker} vs {market_ticker}")
    print(f"   Risk-free rate: {risk_free_rate:.1%} annual")
    
    # Calculate date range
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=period_days + 50)).strftime('%Y-%m-%d')
    
    # Convert annual risk-free rate to daily
    risk_free_daily = risk_free_rate / 252
    
    # Get price data
    stock_df = get_stock_prices_fmp(stock_ticker, start_date, end_date, api_key)
    market_df = get_stock_prices_fmp(market_ticker, start_date, end_date, api_key)
    
    if stock_df.empty or market_df.empty:
        return {
            "uptrend_alpha": None,
            "uptrend_beta": None,
            "downtrend_alpha": None,
            "downtrend_beta": None,
            "error": "Failed to fetch price data"
        }
    
    # Align dates and calculate returns
    merged_df = stock_df.merge(market_df, on='date', suffixes=('_stock', '_market'))
    merged_df = merged_df.sort_values('date').reset_index(drop=True)
    merged_df = merged_df.tail(period_days).reset_index(drop=True)
    
    # Calculate daily returns
    merged_df['return_stock'] = merged_df['close_stock'].pct_change()
    merged_df['return_market'] = merged_df['close_market'].pct_change()
    
    # Remove NaN values
    merged_df = merged_df.dropna()
    
    # Calculate excess returns (subtract risk-free rate)
    merged_df['excess_return_stock'] = merged_df['return_stock'] - risk_free_daily
    merged_df['excess_return_market'] = merged_df['return_market'] - risk_free_daily
    
    # Separate uptrend and downtrend periods (based on market excess returns)
    uptrend_mask = merged_df['excess_return_market'] > 0
    downtrend_mask = merged_df['excess_return_market'] < 0
    
    uptrend_data = merged_df[uptrend_mask]
    downtrend_data = merged_df[downtrend_mask]
    
    def calculate_single_beta(data):
        """Calculate single beta for a specific period using excess returns"""
        if len(data) < 5:
            return None, None
        
        # Single regression: R_stock - Rf = α + β * (R_market - Rf)
        slope, intercept, _, _, _ = stats.linregress(
            data['excess_return_market'], data['excess_return_stock']
        )
        
        alpha = intercept * 252        # Annualized alpha (risk-adjusted)
        beta = slope                   # Beta to market
        
        return alpha, beta
    
    # Calculate for uptrend and downtrend
    uptrend_alpha, uptrend_beta = calculate_single_beta(uptrend_data)
    downtrend_alpha, downtrend_beta = calculate_single_beta(downtrend_data)
    
    # Return simple results
    results = {
        "uptrend_alpha": uptrend_alpha,
        "uptrend_beta": uptrend_beta,
        "downtrend_alpha": downtrend_alpha,
        "downtrend_beta": downtrend_beta,
        "risk_free_rate": risk_free_rate
    }
    
    print(f"📊 Single Beta Results:")
    print(f"   Uptrend:   α={uptrend_alpha:.4f}, β={uptrend_beta:.4f}")
    print(f"   Downtrend: α={downtrend_alpha:.4f}, β={downtrend_beta:.4f}")
    
    return results

In [9]:
sector_filter = sector_index
double_beta_result = orthogonalize_sector_analysis_three_params(ticker, sector_filter, "SPY", 252)
double_beta_result

�� Orthogonal Sector Analysis: MSFT vs SPY + XLK
   Risk-free rate: 2.5% annual
   ✅ Results: α=0.1056, β_market=0.8943, β_sector=0.5856, R²=0.5872


{'alpha': np.float64(0.10559582181615206),
 'market_beta': np.float64(0.8942812766222725),
 'sector_beta': np.float64(0.5856184442871407),
 'r_squared': np.float64(0.5872069009078389),
 'risk_free_rate': 0.025,
 'data_points': 206}

In [10]:
"""
Step 1-2: Get Micro + Macro Factors + Date Ranges - No Sector
"""

import json
import redis
import re
from typing import Any, Dict, Union, List
from pathlib import Path
import sys

# Add project root to path
ROOT_SENTINEL = "LLM_Call_Agent.py"
def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / ROOT_SENTINEL).exists():
            return path
    raise FileNotFoundError(f"Could not locate {ROOT_SENTINEL} upward from {start}")

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from LLM_Call_Agent import LLMCallAgent
from langchain.output_parsers import PydanticOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

# Redis Configuration
REDIS_HOST = "redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com"
REDIS_PORT = 16376
REDIS_USERNAME = "default"
REDIS_PASSWORD = "rl8242B4UItBhFzgHW5APEqZnkYoaEZv"
COLLECTION_NAME = "Stock_Trend_INFOS"

# =============================================================================
# STEP 1: GET MICRO + MACRO FACTORS (NO SECTOR)
# =============================================================================

class FactorSet(BaseModel):
    factor_1: str = Field(description="Keyword for the top catalyst")
    factor_2: str = Field(description="Keyword for the second catalyst.")
    factor_3: str = Field(description="Keyword for the third catalyst.")
    factor_4: str = Field(description="Keyword for the fourth catalyst.")
    factor_5: str = Field(description="Keyword for the fifth catalyst.")
    factor_6: str = Field(description="Keyword for the sixth catalyst.")
    factor_7: str = Field(description="Keyword for the seventh catalyst.")
    factor_8: str = Field(description="Keyword for the eighth catalyst.")
    factor_9: str = Field(description="Keyword for the ninth catalyst.")
    factor_10: str = Field(description="Keyword for the tenth catalyst.")
    factor_11: str = Field(description="Keyword for the eleventh catalyst.")
    factor_12: str = Field(description="Keyword for the twelfth catalyst.")
    factor_13: str = Field(description="Keyword for the thirteenth catalyst.")
    factor_14: str = Field(description="Keyword for the fourteenth catalyst.")
    factor_15: str = Field(description="Keyword for the fifteenth catalyst.")
    factor_16: str = Field(description="Keyword for the sixteenth catalyst.")
    
  
class FactorPayload(BaseModel):
    ticker: str = Field(description="Ticker symbol in uppercase.")
    macro: FactorSet = Field(description="Macro-level catalyst keywords.")
    micro: FactorSet = Field(description="Company-level catalyst keywords.")

# LangChain parser expects Pydantic v2's model_json_schema; shim it for v1.
FactorSet.model_json_schema = classmethod(lambda cls: cls.schema())
FactorPayload.model_json_schema = classmethod(lambda cls: cls.schema())

parser = PydanticOutputParser(pydantic_object=FactorPayload)

def get_system_instructions(language: str = "English") -> str:
    """Get system instructions with language support."""
    base_instructions = f"""
You are a senior equity strategist. Given the supplied stock intelligence and historical trends,
list the most impactful catalysts as short keyword-style names.

Rules:
- Output ONLY canonical event keywords (e.g., "Fed Rate Cut", "China Tariff Hike", "AI Chip Shortage").
- No directions, adjectives, or explanations—just the name of the catalyst.
- Keep each keyword under 60 characters.
- Ground every keyword in the provided context or widely known facts; never invent events.
- List as many as possible, as segmentation as possible (no limit)
- For Micro, make the event general (like iphone integrate with AI, -> product AI innovation)
- I want each factors both negative and positive direction (ex: if put fed rate cut , also put fed rate hike, negative guidence)
- must able to have a postive or negative tune on it 
- Focus on MACRO (market-wide) and MICRO/Sector factors only
- I dont want unclear catalyst name, I want (ex: tune + catalyst, ex: bad/lower-than-expect earning)
- I want Macro + Micro + Sector factors
- Micro include: Business model, Product, Technology, Management, etc. (everything that can impact the company revenue)
- Importance:I want all factors in pair, where I could have the downside and upside (like fed rate cute + fed rate hike) add label on it (liek fed:xx, xx), later will use other pipeline to map this key out
- At the end of the factors: I want to see the factor name (ex: trump win = trump impaction), I want an catalyst name in term of impaction factors not a specific tune 
- 
- Notice: Smart Enought to Capture the Factor, the real driven factor, not just surface one :(Earning beat but worse guidence -> the factor is worse guidence, etcc)
- NOtice: Smart select, if a macro factor is appearance impact more then micro, then select macro (Hint: Good Micro Postive, but then A negaitive Macro, and a downtrend, then Macro is the drive, and shoulr ingore the mciro))
{parser.get_format_instructions()}
""".strip()
    
    if language.lower() != "english":
        language_instruction = f"\n\nIMPORTANT: Output ALL factor names in {language} language only. Do NOT use English."
        return base_instructions + language_instruction
    else:
        return base_instructions

def _extract_json_payload(raw: str) -> str:
    """Strip markdown fences and clamp to the outermost JSON braces."""
    cleaned = raw.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start : end + 1]
    return cleaned

def build_factor_prompt(
    ticker: str, 
    read_information: Union[str, Dict[str, Any]],
    language: str = "English"
) -> str:
    """Human prompt body sent to the LLM."""
    if isinstance(read_information, (dict, list)):
        serialized_context = json.dumps(read_information, ensure_ascii=False, indent=2)
    else:
        serialized_context = str(read_information)

    base_prompt = (
        f"Ticker: {ticker}\n\n"
        "Stock intelligence snapshot:\n"
        f"{serialized_context}\n\n"
        "Task: provide MACRO and MICRO catalyst keywords that my pipeline will map directly. "
        "MACRO: market-wide economic/political events. MICRO: company-specific events or Sector-specific events. "
        "Do NOT include sector/industry factors. "
        "Return only the canonical event names."
    )
    
    if language.lower() != "english":
        language_instruction = f"\n\nIMPORTANT: Output ALL factor names in {language} language only. Do NOT use English."
        return base_prompt + language_instruction
    else:
        return base_prompt

def generate_stock_factors(
    ticker: str,
    read_information: Union[str, Dict[str, Any]],
    provider: str = "deepseek",
    model_override: str | None = None,
    temperature: float = 0.1,
    max_tokens: int = 700,  # Reduced since we only need macro + micro
    language: str = "English"
) -> FactorPayload:
    """Call the LLM and parse keyword factors via LangChain."""
    prompt = build_factor_prompt(ticker, read_information, language)
    system_instructions = get_system_instructions(language)

    if provider == "deepseek":
        model = model_override or "deepseek-chat"
    else:
        provider = "openai"
        model = model_override or "gpt-4o"

    llm_agent = LLMCallAgent(default_provider=provider, default_model=model)

    if provider == "deepseek":
        raw_response = llm_agent.call_deepseek(
            prompt=prompt,
            system_message=system_instructions,
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
        )
    else:
        raw_response = llm_agent.call_openai(
            prompt=prompt,
            system_message=system_instructions,
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
        )

    if not raw_response:
        raise ValueError("Empty response from LLM")

    cleaned = _extract_json_payload(raw_response)
    try:
        return parser.parse(cleaned)
    except Exception as exc:
        raise ValueError(f"LLM response could not be parsed:\n{raw_response}") from exc

def quant_market_expectation_read_agent(ticker: str):
    """Read stock trend data from Redis - MVP version (no async)"""
    redis_key = f"{COLLECTION_NAME}:{ticker.upper()}_trends"

    client = redis.Redis(
        host=REDIS_HOST,
        port=REDIS_PORT,
        username=REDIS_USERNAME,
        password=REDIS_PASSWORD,
        decode_responses=True,
    )

    data = client.get(redis_key)
    if data is None:
        print(f"No stock trend payload stored for {ticker}")
        return None

    return json.loads(data)

# =============================================================================
# STEP 2: GET MICRO + MACRO DATE RANGES (NO SECTOR)
# =============================================================================

class DateRangePayload(BaseModel):
    ticker: str = Field(description="Ticker symbol in uppercase")
    macro: Dict[str, List[List[str]]] = Field(description="Macro factor to date ranges mapping")
    micro: Dict[str, List[List[str]]] = Field(description="Micro factor to date ranges mapping")

# LangChain parser expects Pydantic v2's model_json_schema; shim it for v1.
DateRangePayload.model_json_schema = classmethod(lambda cls: cls.schema())

date_range_parser = PydanticOutputParser(pydantic_object=DateRangePayload)

def get_date_range_system_instructions(language: str = "English") -> str:
    """Get system instructions for date range mapping."""
    base_instructions = f"""
You are a meticulous financial analyst. Given the stock intelligence and historical trends,
map each factor to specific date ranges when those events actually occurred.

Rules:
- For each factor, provide actual date ranges when the event happened
- Use format: ["YYYY-MM-DD", "YYYY-MM-DD"] for each date range
- A factor can have multiple date ranges (e.g., Fed Rate Cut happened multiple times)
- Only use dates that actually exist in the historical data
- If no specific dates are available, provide empty list []
- Focus on major events that would impact stock price
- Be conservative - only include dates you're confident about
- IMPORTANT: You MUST include macro and micro sections (NO sector)
- IMPORTANT: Keep response concise to avoid truncation 
- NOtice: Smart select, if a macro factor is appearance impact more then micro, then select macro (Hint: Good Micro Postive, but then A negaitive Macro, and a downtrend, then Macro is the drive, and shoulr ingore the mciro date range sample at the same date))
{date_range_parser.get_format_instructions()}
""".strip()
    
    if language.lower() != "english":
        language_instruction = f"\n\nIMPORTANT: Output ALL content in {language} language only. Do NOT use English."
        return base_instructions + language_instruction
    else:
        return base_instructions

def build_date_range_prompt(
    ticker: str,
    factor_payload: FactorPayload,
    read_information: Dict[str, Any],
    language: str = "English"
) -> str:
    """Build prompt for date range mapping."""
    factor_summary = json.dumps(factor_payload.dict(), indent=2, ensure_ascii=False)
    
    # Extract historical trends for context
    historical_trends = read_information.get("historical_trends", {})
    trend_summary = {}
    
    for trend_key, trend_data in historical_trends.items():
        trend_summary[trend_key] = {
            "summary": trend_data.get("summary", ""),
            "time_period": trend_data.get("time", {}),
            "macro_reason": trend_data.get("macro_reason", ""),
            "micro_reason": trend_data.get("micro_reason", "")
        }
    
    trend_context = json.dumps(trend_summary, indent=2, ensure_ascii=False)
    
    base_prompt = (
        f"Ticker: {ticker}\n\n"
        f"Factor keywords (macro/micro):\n{factor_summary}\n\n"
        "Historical trend context (with dates and reasons):\n"
        f"{trend_context}\n\n"
        
        "Task: For each factor, map to specific date ranges when those events occurred. "
        "Use the historical trend data to identify actual dates. "
        "Return date ranges in format: [\"start_date\", \"end_date\"]"
        "IMPORTANT: Include macro and micro sections (NO sector) and keep response concise."
    )
    
    if language.lower() != "english":
        language_instruction = f"\n\nIMPORTANT: Output ALL content in {language} language only. Do NOT use English."
        return base_prompt + language_instruction
    else:
        return base_prompt

def map_factors_to_date_ranges(
    ticker: str,
    factor_payload: FactorPayload,
    read_information: Dict[str, Any],
    provider: str = "deepseek",
    model_override: str | None = None,
    temperature: float = 0.0,
    max_tokens: int = 1800,  # Reduced since we only need macro + micro
    language: str = "English"
) -> DateRangePayload:
    """Map factors to date ranges using LLM."""
    prompt = build_date_range_prompt(ticker, factor_payload, read_information, language)
    system_instructions = get_date_range_system_instructions(language)

    if provider == "deepseek":
        model = model_override or "deepseek-chat"
    else:
        provider = "openai"
        model = model_override or "gpt-4o"

    llm_agent = LLMCallAgent(default_provider=provider, default_model=model)

    if provider == "deepseek":
        raw_response = llm_agent.call_deepseek(
            prompt=prompt,
            system_message=system_instructions,
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
        )
    else:
        raw_response = llm_agent.call_openai(
            prompt=prompt,
            system_message=system_instructions,
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
        )

    if not raw_response:
        raise ValueError("Empty response from LLM during date range mapping")

    cleaned = raw_response.strip().strip("`")
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start : end + 1]

    # Check if response is complete
    if not cleaned.endswith("}"):
        print(f"⚠️ Response may be truncated. Last 100 chars: {cleaned[-100:]}")
        # Try to fix incomplete JSON
        if '"micro"' not in cleaned:
            cleaned += ', "micro": {}}'
        elif '"macro"' not in cleaned:
            cleaned += ', "macro": {}}'

    try:
        return date_range_parser.parse(cleaned)
    except Exception as exc:
        print(f"❌ Raw response: {raw_response}")
        print(f"❌ Cleaned response: {cleaned}")
        raise ValueError(f"Date range mapping response could not be parsed:\n{raw_response}") from exc

# =============================================================================
# STEP 1-2 USAGE FUNCTIONS (MICRO + MACRO ONLY)
# =============================================================================

def step1_get_factors(ticker: str, language: str = "English"):
    """Step 1: Get micro + macro factors (no sector)"""
    print(f" Step 1: Getting MICRO + MACRO factors for {ticker}")
    
    # Read stock intelligence from Redis
    read_information = quant_market_expectation_read_agent(ticker)
    if not read_information:
        raise ValueError(f"No stock intelligence found for {ticker}")
    
    # Generate factors using LLM
    factor_result = generate_stock_factors(
        ticker=ticker, 
        read_information=read_information,
        language=language
    )
    
    # Extract factor lists
    macro_factors = [
        factor_result.macro.factor_1,
        factor_result.macro.factor_2,
        factor_result.macro.factor_3,
        factor_result.macro.factor_4,
        factor_result.macro.factor_5,
        factor_result.macro.factor_6,
        factor_result.macro.factor_7,
        factor_result.macro.factor_8,
        factor_result.macro.factor_9,
        factor_result.macro.factor_10,
        factor_result.macro.factor_11,
        factor_result.macro.factor_12,
        factor_result.macro.factor_13,
        factor_result.macro.factor_14,
        factor_result.macro.factor_15,
        factor_result.macro.factor_16,
     
    ]
    
    micro_factors = [
        factor_result.micro.factor_1,
        factor_result.micro.factor_2,
        factor_result.micro.factor_3,
        factor_result.micro.factor_4,
        factor_result.micro.factor_5,
        factor_result.micro.factor_6,
        factor_result.micro.factor_7,
        factor_result.micro.factor_8,
        factor_result.micro.factor_9,
        factor_result.micro.factor_10,
        factor_result.micro.factor_11,
        factor_result.micro.factor_12,
        factor_result.micro.factor_13,
        factor_result.micro.factor_14,
        factor_result.micro.factor_15,
        factor_result.micro.factor_16,
  
    ]
    
    print(f"✅ Generated {len(macro_factors)} macro factors")
    print(f"✅ Generated {len(micro_factors)} micro factors")
    
    return {
        "factor_payload": factor_result,
        "macro_factors": macro_factors,
        "micro_factors": micro_factors,
        "read_information": read_information
    }

def step2_get_date_ranges(ticker: str, factor_result: Dict[str, Any], language: str = "English"):
    """Step 2: Map micro + macro factors to date ranges"""
    print(f"️ Step 2: Mapping MICRO + MACRO factors to date ranges for {ticker}")
    
    factor_payload = factor_result["factor_payload"]
    read_information = factor_result["read_information"]
    
    # Map factors to date ranges using LLM
    date_range_result = map_factors_to_date_ranges(
        ticker=ticker,
        factor_payload=factor_payload,
        read_information=read_information,
        language=language
    )
    
    print(f"✅ Mapped macro factors to date ranges")
    print(f"✅ Mapped micro factors to date ranges")
    
    return date_range_result



/Users/xikinki/anaconda3/envs/arviz_env/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3577: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)


In [11]:
step1_result = step1_get_factors(ticker, language)
step2_result = step2_get_date_ranges(ticker, step1_result, language)
        

 Step 1: Getting MICRO + MACRO factors for MSFT
✅ Generated 16 macro factors
✅ Generated 16 micro factors
️ Step 2: Mapping MICRO + MACRO factors to date ranges for MSFT
✅ Mapped macro factors to date ranges
✅ Mapped micro factors to date ranges


## Double Beta + Self-Variance Filter System

In [12]:
"""
Step 3: Beta Filtering - Get Real Market Data and Calculate Beta-Adjusted Impact
DAILY RETURNS: Use daily returns throughout
"""

import numpy as np
import pandas as pd
import requests
from datetime import datetime, timedelta
from typing import Dict, List, Any

# Your FMP API key
FMP_API_KEY = "9dfbbfa29d93f4793f246e8fb5ca5e74"

def get_stock_prices_fmp(ticker: str, start_date: str, end_date: str, api_key: str = None) -> pd.DataFrame:
    """Get stock prices from FMP API"""
    if api_key is None:
        api_key = FMP_API_KEY
    
    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{ticker}"
    params = {
        'from': start_date,
        'to': end_date,
        'apikey': api_key
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if 'historical' in data:
            df = pd.DataFrame(data['historical'])
            df['date'] = pd.to_datetime(df['date'])
            df = df.sort_values('date').reset_index(drop=True)
            return df[['date', 'close']]
        else:
            print(f"❌ No historical data found for {ticker}")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"❌ Error fetching data for {ticker}: {e}")
        return pd.DataFrame()

def get_risk_free_rate(start_date: str, end_date: str) -> float:
    """Get risk-free rate for the period (simplified - using constant 2.5%)"""
    # In practice, you'd fetch actual Treasury rates
    # For MVP, using constant 2.5% annual
    return 0.025

def calculate_annual_volatility(ticker: str, start_date: str, end_date: str) -> float:
    """Calculate annual volatility for the stock"""
    try:
        # Get stock price data
        stock_df = get_stock_prices_fmp(ticker, start_date, end_date)
        
        if stock_df.empty:
            print(f"❌ No price data found for {ticker} volatility calculation")
            return 0.0
        
        # Calculate daily returns
        stock_df['daily_return'] = stock_df['close'].pct_change()
        
        # Remove NaN values
        stock_df = stock_df.dropna()
        
        if len(stock_df) < 10:
            print(f"❌ Insufficient data for volatility calculation ({len(stock_df)} days)")
            return 0.0
        
        # Calculate daily volatility
        daily_volatility = stock_df['daily_return'].std()
        
        # Annualize using 252 trading days
        annual_volatility = daily_volatility * np.sqrt(252)
        
        print(f"   📊 Annual Volatility for {ticker}: {annual_volatility:.4f} ({annual_volatility*100:.2f}%)")
        
        return annual_volatility
        
    except Exception as e:
        print(f"❌ Error calculating volatility for {ticker}: {e}")
        return 0.0

def map_date_range_to_trend_data(start_date: str, end_date: str, historical_trends: Dict[str, Any]) -> Dict[str, float]:
    """
    Map a date range to the corresponding trend data using exact key matching
    Returns stock return rate and SPY return rate for the period
    """
    # Create the exact period key format that matches your database
    period_key = f"{start_date} to {end_date}"
    
    # Look for exact match in trend periods
    for trend_key, trend_data in historical_trends.items():
        trend_period = trend_data.get('current', '')  # This contains "2025-07-07 to 2025-07-23"
        
        if period_key == trend_period:
            stock_return = trend_data.get('day average_return', 0.0)
            spy_return = trend_data.get('SPY_return_rate', 0.0)
            
            # Handle None values
            if stock_return is None:
                stock_return = 0.0
            if spy_return is None:
                spy_return = 0.0
                
            return {
                "stock_daily_return": stock_return,
                "spy_daily_return": spy_return,
                "trend_key": trend_key
            }
    
    # If no exact match found, return zeros
    print(f"⚠️ No exact trend data found for period {period_key}")
    return {
        "stock_daily_return": 0.0,
        "spy_daily_return": 0.0,
        "trend_key": "no_match"
    }

def step3_beta_filtering(ticker: str, step2_result: Any, read_information: Dict[str, Any], 
                        market_ticker: str = "SPY", risk_free_rate: float = 0.025) -> Dict[str, Any]:
    """
    Step 3: Beta Filtering - Calculate real market data and beta-adjusted impact
    DAILY RETURNS: Use daily returns throughout
    NOW USES SPY_RETURN_RATE from stock trend database instead of fetching SPY data
    INCLUDES VOLATILITY NORMALIZATION using annual_volatility / 15.874
    
    Args:
        ticker: Stock ticker (e.g., 'UNH')
        step2_result: Date ranges from Step 2
        read_information: Stock trend data from Step 1 (contains historical_trends)
        market_ticker: Market benchmark (default: 'SPY') - not used anymore
        risk_free_rate: Annual risk-free rate (default: 2.5%)
    
    Returns:
        dict: Beta-filtered factor impacts with volatility normalization
    """
    print(f"🔍 Step 3: Beta filtering for {ticker}")
    print(f"   Using SPY_return_rate from stock trend database")
    
    # Get market beta from orthogonal sector analysis
    # Assuming you have this from your previous analysis
    market_beta = 1.2777  # This should come from your orthogonal analysis
    
    print(f"   Using market beta: {market_beta:.4f}")
    print(f"   Risk-free rate: {risk_free_rate:.1%} annual")
    
    # Extract historical trends from read_information
    historical_trends = read_information.get('historical_trends', {})
    if not historical_trends:
        print("❌ No historical trends found in read_information")
        return {"error": "No historical trends found"}
    
    print(f"   Found {len(historical_trends)} historical trend periods")
    
    # Get date ranges for macro and micro factors
    macro_date_ranges = step2_result.macro
    micro_date_ranges = step2_result.micro
    
    # Calculate date range for fetching data (get all dates needed)
    all_dates = []
    for factor_ranges in macro_date_ranges.values():
        all_dates.extend(factor_ranges)
    for factor_ranges in micro_date_ranges.values():
        all_dates.extend(factor_ranges)
    
    if not all_dates:
        print("❌ No date ranges found")
        return {"error": "No date ranges found"}
    
    # Get overall date range
    min_date = min([min(date_range) for date_range in all_dates if date_range])
    max_date = max([max(date_range) for date_range in all_dates if date_range])
    
    print(f"   Processing date ranges from {min_date} to {max_date}")
    
    # Calculate annual volatility for the stock
    annual_volatility = calculate_annual_volatility(ticker, min_date, max_date)
    volatility_factor = annual_volatility / 15.874
    
    print(f"   🔧 Volatility Factor (vol/15.874): {volatility_factor:.4f}")
    
    # Calculate risk-free rate for the period
    risk_free_rate_period = get_risk_free_rate(min_date, max_date)
    
    # Process macro factors
    macro_results = {}
    print(f"\n Processing {len(macro_date_ranges)} macro factors...")
    
    for factor_name, date_ranges in macro_date_ranges.items():
        if not date_ranges:  # Skip empty date ranges
            continue
            
        factor_impacts = []
        
        for start_date, end_date in date_ranges:
            # Get stock and SPY returns from historical trends
            trend_data = map_date_range_to_trend_data(start_date, end_date, historical_trends)
            stock_daily_return = trend_data["stock_daily_return"]
            spy_daily_return = trend_data["spy_daily_return"]
            
            # Calculate risk-free return for this period (daily)
            days = (datetime.strptime(end_date, '%Y-%m-%d') - datetime.strptime(start_date, '%Y-%m-%d')).days
            risk_free_daily = risk_free_rate_period / 365  # Daily risk-free rate
            
            # Calculate beta-adjusted macro impact
            real_macro_impact = market_beta * (spy_daily_return - risk_free_daily) + risk_free_daily
            
            # Calculate micro impact (what's left after macro)
            real_micro_impact = stock_daily_return - real_macro_impact
            
            # Apply volatility normalization to micro impact
            if real_micro_impact > 0:
                # If real micro impact is positive: micro - volatility/15.874
                normalized_micro_impact = real_micro_impact - volatility_factor
            else:
                # If real micro impact is negative: micro + volatility/15.874
                normalized_micro_impact = real_micro_impact + volatility_factor
            
            factor_impacts.append({
                "period": f"{start_date} to {end_date}",
                "days": days,
                "stock_daily_return": stock_daily_return,  # DAILY return from trend data
                "spy_daily_return": spy_daily_return,      # DAILY return from trend data
                "risk_free_daily": risk_free_daily,
                "real_macro_impact": real_macro_impact,
                "real_micro_impact": real_micro_impact,
                "normalized_micro_impact": normalized_micro_impact,
                "volatility_factor": volatility_factor,
                "trend_key": trend_data["trend_key"]
            })
        
        macro_results[factor_name] = factor_impacts
        print(f"   ✅ {factor_name}: {len(factor_impacts)} periods")
    
    # Process micro factors
    micro_results = {}
    print(f"\n Processing {len(micro_date_ranges)} micro factors...")
    
    for factor_name, date_ranges in micro_date_ranges.items():
        if not date_ranges:  # Skip empty date ranges
            continue
            
        factor_impacts = []
        
        for start_date, end_date in date_ranges:
            # Get stock and SPY returns from historical trends
            trend_data = map_date_range_to_trend_data(start_date, end_date, historical_trends)
            stock_daily_return = trend_data["stock_daily_return"]
            spy_daily_return = trend_data["spy_daily_return"]
            
            # Calculate risk-free return for this period (daily)
            days = (datetime.strptime(end_date, '%Y-%m-%d') - datetime.strptime(start_date, '%Y-%m-%d')).days
            risk_free_daily = risk_free_rate_period / 365  # Daily risk-free rate
            
            # Calculate beta-adjusted macro impact
            real_macro_impact = market_beta * (spy_daily_return - risk_free_daily) + risk_free_daily
            
            # Calculate micro impact (what's left after macro)
            real_micro_impact = stock_daily_return - real_macro_impact
            
            # Apply volatility normalization to micro impact
            if real_micro_impact > 0:
                # If real micro impact is positive: micro - volatility/15.874
                normalized_micro_impact = real_micro_impact - volatility_factor
            else:
                # If real micro impact is negative: micro + volatility/15.874
                normalized_micro_impact = real_micro_impact + volatility_factor
            
            factor_impacts.append({
                "period": f"{start_date} to {end_date}",
                "days": days,
                "stock_daily_return": stock_daily_return,  # DAILY return from trend data
                "spy_daily_return": spy_daily_return,      # DAILY return from trend data
                "risk_free_daily": risk_free_daily,
                "real_macro_impact": real_macro_impact,
                "real_micro_impact": real_micro_impact,
                "normalized_micro_impact": normalized_micro_impact,
                "volatility_factor": volatility_factor,
                "trend_key": trend_data["trend_key"]
            })
        
        micro_results[factor_name] = factor_impacts
        print(f"   ✅ {factor_name}: {len(factor_impacts)} periods")
    
    # Calculate weighted averages (using duration as weights)
    def calculate_weighted_averages(factor_results: Dict[str, List[Dict]]) -> Dict[str, Dict[str, float]]:
        """Calculate weighted averages for each factor"""
        weighted_results = {}
        
        for factor_name, impacts in factor_results.items():
            if not impacts:
                continue
                
            total_duration = sum(impact['days'] for impact in impacts)
            weighted_macro_sum = sum(impact['days'] * impact['real_macro_impact'] for impact in impacts)
            weighted_micro_sum = sum(impact['days'] * impact['real_micro_impact'] for impact in impacts)
            weighted_normalized_micro_sum = sum(impact['days'] * impact['normalized_micro_impact'] for impact in impacts)
            
            weighted_results[factor_name] = {
                "weighted_macro_impact": weighted_macro_sum / total_duration if total_duration > 0 else 0,
                "weighted_micro_impact": weighted_micro_sum / total_duration if total_duration > 0 else 0,
                "weighted_normalized_micro_impact": weighted_normalized_micro_sum / total_duration if total_duration > 0 else 0,
                "total_duration": total_duration,
                "periods": len(impacts)
            }
        
        return weighted_results
    
    macro_weighted = calculate_weighted_averages(macro_results)
    micro_weighted = calculate_weighted_averages(micro_results)
    
    print(f"\n✅ Beta filtering completed!")
    print(f"   Processed {len(macro_results)} macro factors")
    print(f"   Processed {len(micro_results)} micro factors")
    print(f"   Used SPY_return_rate from stock trend database")
    print(f"   Applied volatility normalization: {volatility_factor:.4f}")
    
    return {
        "ticker": ticker,
        "market_beta": market_beta,
        "risk_free_rate": risk_free_rate,
        "annual_volatility": annual_volatility,
        "volatility_factor": volatility_factor,
        "macro_results": macro_results,
        "micro_results": micro_results,
        "macro_weighted": macro_weighted,
        "micro_weighted": micro_weighted,
        "data_period": f"{min_date} to {max_date}"
    }

In [13]:
# """

### Old Method Does not Applicable
########################################################################
# Step 3: Beta Filtering - Get Real Market Data and Calculate Beta-Adjusted Impact
# DAILY RETURNS: Use daily returns throughout
# """

# import numpy as np
# import pandas as pd
# import requests
# from datetime import datetime, timedelta
# from typing import Dict, List, Any

# # Your FMP API key
# FMP_API_KEY = "9dfbbfa29d93f4793f246e8fb5ca5e74"

# def get_stock_prices_fmp(ticker: str, start_date: str, end_date: str, api_key: str = None) -> pd.DataFrame:
#     """Get stock prices from FMP API"""
#     if api_key is None:
#         api_key = FMP_API_KEY
    
#     url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{ticker}"
#     params = {
#         'from': start_date,
#         'to': end_date,
#         'apikey': api_key
#     }
    
#     try:
#         response = requests.get(url, params=params)
#         response.raise_for_status()
#         data = response.json()
        
#         if 'historical' in data:
#             df = pd.DataFrame(data['historical'])
#             df['date'] = pd.to_datetime(df['date'])
#             df = df.sort_values('date').reset_index(drop=True)
#             return df[['date', 'close']]
#         else:
#             print(f"❌ No historical data found for {ticker}")
#             return pd.DataFrame()
            
#     except Exception as e:
#         print(f"❌ Error fetching data for {ticker}: {e}")
#         return pd.DataFrame()

# def get_risk_free_rate(start_date: str, end_date: str) -> float:
#     """Get risk-free rate for the period (simplified - using constant 2.5%)"""
#     # In practice, you'd fetch actual Treasury rates
#     # For MVP, using constant 2.5% annual
#     return 0.025

# def map_date_range_to_trend_data(start_date: str, end_date: str, historical_trends: Dict[str, Any]) -> Dict[str, float]:
#     """
#     Map a date range to the corresponding trend data using exact key matching
#     Returns stock return rate and SPY return rate for the period
#     """
#     # Create the exact period key format that matches your database
#     period_key = f"{start_date} to {end_date}"
    
#     # Look for exact match in trend periods
#     for trend_key, trend_data in historical_trends.items():
#         trend_period = trend_data.get('current', '')  # This contains "2025-07-07 to 2025-07-23"
        
#         if period_key == trend_period:
#             stock_return = trend_data.get('day average_return', 0.0)
#             spy_return = trend_data.get('SPY_return_rate', 0.0)
            
#             # Handle None values
#             if stock_return is None:
#                 stock_return = 0.0
#             if spy_return is None:
#                 spy_return = 0.0
                
#             return {
#                 "stock_daily_return": stock_return,
#                 "spy_daily_return": spy_return,
#                 "trend_key": trend_key
#             }
    
#     # If no exact match found, return zeros
#     print(f"⚠️ No exact trend data found for period {period_key}")
#     return {
#         "stock_daily_return": 0.0,
#         "spy_daily_return": 0.0,
#         "trend_key": "no_match"
#     }

# def step3_beta_filtering(ticker: str, step2_result: Any, read_information: Dict[str, Any], 
#                         market_ticker: str = "SPY", risk_free_rate: float = 0.025) -> Dict[str, Any]:
#     """
#     Step 3: Beta Filtering - Calculate real market data and beta-adjusted impact
#     DAILY RETURNS: Use daily returns throughout
#     NOW USES SPY_RETURN_RATE from stock trend database instead of fetching SPY data
    
#     Args:
#         ticker: Stock ticker (e.g., 'UNH')
#         step2_result: Date ranges from Step 2
#         read_information: Stock trend data from Step 1 (contains historical_trends)
#         market_ticker: Market benchmark (default: 'SPY') - not used anymore
#         risk_free_rate: Annual risk-free rate (default: 2.5%)
    
#     Returns:
#         dict: Beta-filtered factor impacts
#     """
#     print(f"🔍 Step 3: Beta filtering for {ticker}")
#     print(f"   Using SPY_return_rate from stock trend database")
    
#     # Get market beta from orthogonal sector analysis
#     # Assuming you have this from your previous analysis
#     market_beta = 1.2777  # This should come from your orthogonal analysis
    
#     print(f"   Using market beta: {market_beta:.4f}")
#     print(f"   Risk-free rate: {risk_free_rate:.1%} annual")
    
#     # Extract historical trends from read_information
#     historical_trends = read_information.get('historical_trends', {})
#     if not historical_trends:
#         print("❌ No historical trends found in read_information")
#         return {"error": "No historical trends found"}
    
#     print(f"   Found {len(historical_trends)} historical trend periods")
    
#     # Get date ranges for macro and micro factors
#     macro_date_ranges = step2_result.macro
#     micro_date_ranges = step2_result.micro
    
#     # Calculate date range for fetching data (get all dates needed)
#     all_dates = []
#     for factor_ranges in macro_date_ranges.values():
#         all_dates.extend(factor_ranges)
#     for factor_ranges in micro_date_ranges.values():
#         all_dates.extend(factor_ranges)
    
#     if not all_dates:
#         print("❌ No date ranges found")
#         return {"error": "No date ranges found"}
    
#     # Get overall date range
#     min_date = min([min(date_range) for date_range in all_dates if date_range])
#     max_date = max([max(date_range) for date_range in all_dates if date_range])
    
#     print(f"   Processing date ranges from {min_date} to {max_date}")
    
#     # Calculate risk-free rate for the period
#     risk_free_rate_period = get_risk_free_rate(min_date, max_date)
    
#     # Process macro factors
#     macro_results = {}
#     print(f"\n Processing {len(macro_date_ranges)} macro factors...")
    
#     for factor_name, date_ranges in macro_date_ranges.items():
#         if not date_ranges:  # Skip empty date ranges
#             continue
            
#         factor_impacts = []
        
#         for start_date, end_date in date_ranges:
#             # Get stock and SPY returns from historical trends
#             trend_data = map_date_range_to_trend_data(start_date, end_date, historical_trends)
#             stock_daily_return = trend_data["stock_daily_return"]
#             spy_daily_return = trend_data["spy_daily_return"]
            
#             # Calculate risk-free return for this period (daily)
#             days = (datetime.strptime(end_date, '%Y-%m-%d') - datetime.strptime(start_date, '%Y-%m-%d')).days
#             risk_free_daily = risk_free_rate_period / 365  # Daily risk-free rate
            
#             # Calculate beta-adjusted macro impact
#             real_macro_impact = market_beta * (spy_daily_return - risk_free_daily) + risk_free_daily
            
#             # Calculate micro impact (what's left after macro)
#             real_micro_impact = stock_daily_return - real_macro_impact
            
#             factor_impacts.append({
#                 "period": f"{start_date} to {end_date}",
#                 "days": days,
#                 "stock_daily_return": stock_daily_return,  # DAILY return from trend data
#                 "spy_daily_return": spy_daily_return,      # DAILY return from trend data
#                 "risk_free_daily": risk_free_daily,
#                 "real_macro_impact": real_macro_impact,
#                 "real_micro_impact": real_micro_impact,
#                 "trend_key": trend_data["trend_key"]
#             })
        
#         macro_results[factor_name] = factor_impacts
#         print(f"   ✅ {factor_name}: {len(factor_impacts)} periods")
    
#     # Process micro factors
#     micro_results = {}
#     print(f"\n Processing {len(micro_date_ranges)} micro factors...")
    
#     for factor_name, date_ranges in micro_date_ranges.items():
#         if not date_ranges:  # Skip empty date ranges
#             continue
            
#         factor_impacts = []
        
#         for start_date, end_date in date_ranges:
#             # Get stock and SPY returns from historical trends
#             trend_data = map_date_range_to_trend_data(start_date, end_date, historical_trends)
#             stock_daily_return = trend_data["stock_daily_return"]
#             spy_daily_return = trend_data["spy_daily_return"]
            
#             # Calculate risk-free return for this period (daily)
#             days = (datetime.strptime(end_date, '%Y-%m-%d') - datetime.strptime(start_date, '%Y-%m-%d')).days
#             risk_free_daily = risk_free_rate_period / 365  # Daily risk-free rate
            
#             # Calculate beta-adjusted macro impact
#             real_macro_impact = market_beta * (spy_daily_return - risk_free_daily) + risk_free_daily
            
#             # Calculate micro impact (what's left after macro)
#             real_micro_impact = stock_daily_return - real_macro_impact
            
#             factor_impacts.append({
#                 "period": f"{start_date} to {end_date}",
#                 "days": days,
#                 "stock_daily_return": stock_daily_return,  # DAILY return from trend data
#                 "spy_daily_return": spy_daily_return,      # DAILY return from trend data
#                 "risk_free_daily": risk_free_daily,
#                 "real_macro_impact": real_macro_impact,
#                 "real_micro_impact": real_micro_impact,
#                 "trend_key": trend_data["trend_key"]
#             })
        
#         micro_results[factor_name] = factor_impacts
#         print(f"   ✅ {factor_name}: {len(factor_impacts)} periods")
    
#     # Calculate weighted averages (using duration as weights)
#     def calculate_weighted_averages(factor_results: Dict[str, List[Dict]]) -> Dict[str, Dict[str, float]]:
#         """Calculate weighted averages for each factor"""
#         weighted_results = {}
        
#         for factor_name, impacts in factor_results.items():
#             if not impacts:
#                 continue
                
#             total_duration = sum(impact['days'] for impact in impacts)
#             weighted_macro_sum = sum(impact['days'] * impact['real_macro_impact'] for impact in impacts)
#             weighted_micro_sum = sum(impact['days'] * impact['real_micro_impact'] for impact in impacts)
            
#             weighted_results[factor_name] = {
#                 "weighted_macro_impact": weighted_macro_sum / total_duration if total_duration > 0 else 0,
#                 "weighted_micro_impact": weighted_micro_sum / total_duration if total_duration > 0 else 0,
#                 "total_duration": total_duration,
#                 "periods": len(impacts)
#             }
        
#         return weighted_results
    
#     macro_weighted = calculate_weighted_averages(macro_results)
#     micro_weighted = calculate_weighted_averages(micro_results)
    
#     print(f"\n✅ Beta filtering completed!")
#     print(f"   Processed {len(macro_results)} macro factors")
#     print(f"   Processed {len(micro_results)} micro factors")
#     print(f"   Used SPY_return_rate from stock trend database")
    
#     return {
#         "ticker": ticker,
#         "market_beta": market_beta,
#         "risk_free_rate": risk_free_rate,
#         "macro_results": macro_results,
#         "micro_results": micro_results,
#         "macro_weighted": macro_weighted,
#         "micro_weighted": micro_weighted,
#         "data_period": f"{min_date} to {max_date}"
#     }

In [14]:
step3_result = step3_beta_filtering(ticker, step2_result, step1_result["read_information"], "SPY", 0.025)

🔍 Step 3: Beta filtering for MSFT
   Using SPY_return_rate from stock trend database
   Using market beta: 1.2777
   Risk-free rate: 2.5% annual
   Found 30 historical trend periods
   Processing date ranges from 2024-10-02 to 2025-10-01
   📊 Annual Volatility for MSFT: 0.2468 (24.68%)
   🔧 Volatility Factor (vol/15.874): 0.0155

 Processing 16 macro factors...
⚠️ No exact trend data found for period 2024-12-17 to 2024-12-17
   ✅ Fed Rate Cut: 2 periods
   ✅ Trump Tariff Implementation: 6 periods
   ✅ Trump Tariff Relief: 5 periods
   ✅ Fed Policy Uncertainty: 4 periods
   ✅ Fed Policy Clarity: 5 periods
   ✅ Market Tech Valuation Concerns: 5 periods
   ✅ Market Tech Valuation Support: 4 periods
   ✅ Geopolitical Tensions: 3 periods
   ✅ Inflation Concerns: 4 periods
   ✅ Stagflation Risk: 1 periods
   ✅ Economic Growth Acceleration: 1 periods
   ✅ Government Shutdown Threat: 1 periods

 Processing 16 micro factors...
   ✅ Azure Growth Guidance Miss: 2 periods
   ✅ Azure Growth Guidanc

In [15]:
step3_result

{'ticker': 'MSFT',
 'market_beta': 1.2777,
 'risk_free_rate': 0.025,
 'annual_volatility': np.float64(0.24680514558325914),
 'volatility_factor': np.float64(0.015547760210612268),
 'macro_results': {'Fed Rate Cut': [{'period': '2024-12-17 to 2024-12-17',
    'days': 0,
    'stock_daily_return': 0.0,
    'spy_daily_return': 0.0,
    'risk_free_daily': 6.849315068493152e-05,
    'real_macro_impact': -1.902054794520549e-05,
    'real_micro_impact': 1.902054794520549e-05,
    'normalized_micro_impact': np.float64(-0.015528739662667063),
    'volatility_factor': np.float64(0.015547760210612268),
    'trend_key': 'no_match'},
   {'period': '2025-09-05 to 2025-09-19',
    'days': 14,
    'stock_daily_return': 0.00458,
    'spy_daily_return': 0.0028,
    'risk_free_daily': 6.849315068493152e-05,
    'real_macro_impact': 0.0035585394520547945,
    'real_micro_impact': 0.0010214605479452053,
    'normalized_micro_impact': np.float64(-0.014526299662667063),
    'volatility_factor': np.float64(0.0

## Impaction Metrics

In [16]:
"""
Step 4: Impact Metrics - Generate Final Aggregated Metrics
DAILY RETURNS: Works with daily returns from Step 3
"""

import numpy as np
import pandas as pd
from typing import Dict, List, Any

def step4_impact_metrics(step3_result: Dict[str, Any]) -> Dict[str, Any]:
    """
    Step 4: Generate final impact metrics in Quant Agent format
    
    Args:
        step3_result: Beta filtering results from Step 3 (with daily returns)
    
    Returns:
        dict: Final aggregated metrics matching Quant Agent format
    """
    print(f"🔍 Step 4: Generating impact metrics for {step3_result['ticker']}")
    
    # Extract data from step3_result
    macro_weighted = step3_result['macro_weighted']
    micro_weighted = step3_result['micro_weighted']
    market_beta = step3_result['market_beta']
    risk_free_rate = step3_result['risk_free_rate']
    
    # Convert to Quant Agent format
    aggregated_metrics = {
        "macro": {},
        "micro": {}
    }
    
    # Process macro factors
    print(f"\n Processing {len(macro_weighted)} macro factors...")
    
    for factor_name, factor_data in macro_weighted.items():
        # Calculate variance from the weighted impacts
        macro_impact = factor_data['weighted_macro_impact']
        micro_impact = factor_data['weighted_micro_impact']
        
        # Estimate variance (simplified - in practice you'd calculate from actual data)
        # Using a reasonable estimate based on typical factor volatility
        macro_variance = abs(macro_impact) * 0.1  # 10% of impact as variance estimate
        micro_variance = abs(micro_impact) * 0.15  # 15% of impact as variance estimate
        
        # Convert to Quant Agent format
        aggregated_metrics["macro"][factor_name] = {
            "trend_keys": [f"macro_{factor_name.lower().replace(' ', '_')}"],
            "trend_count": factor_data['periods'],
            "weighted_mean": macro_impact,  # DAILY return
            "weighted_variance": macro_variance,
            "average_duration": factor_data['total_duration'] / factor_data['periods'] if factor_data['periods'] > 0 else 0,
            "total_duration": factor_data['total_duration'],
            "micro_impact": micro_impact,  # Additional field for micro impact
            "micro_variance": micro_variance
        }
        
        print(f"   ✅ {factor_name}: μ={macro_impact:.4f}, σ²={macro_variance:.4f}")
    
    # Process micro factors
    print(f"\n Processing {len(micro_weighted)} micro factors...")
    
    for factor_name, factor_data in micro_weighted.items():
        # Calculate variance from the weighted impacts
        macro_impact = factor_data['weighted_macro_impact']
        micro_impact = factor_data['weighted_micro_impact']
        
        # Estimate variance (simplified - in practice you'd calculate from actual data)
        macro_variance = abs(macro_impact) * 0.1  # 10% of impact as variance estimate
        micro_variance = abs(micro_impact) * 0.15  # 15% of impact as variance estimate
        
        # Convert to Quant Agent format
        aggregated_metrics["micro"][factor_name] = {
            "trend_keys": [f"micro_{factor_name.lower().replace(' ', '_')}"],
            "trend_count": factor_data['periods'],
            "weighted_mean": micro_impact,  # DAILY return
            "weighted_variance": micro_variance,
            "average_duration": factor_data['total_duration'] / factor_data['periods'] if factor_data['periods'] > 0 else 0,
            "total_duration": factor_data['total_duration'],
            "macro_impact": macro_impact,  # Additional field for macro impact
            "macro_variance": macro_variance
        }
        
        print(f"   ✅ {factor_name}: μ={micro_impact:.4f}, σ²={micro_variance:.4f}")
    
    # Generate summary DataFrame (exact same format as Quant Agent)
    summary_data = []
    
    for scope, factors in aggregated_metrics.items():
        for factor_name, factor_data in factors.items():
            summary_data.append({
                "scope": scope,
                "factor": factor_name,
                "trend_keys": ", ".join(factor_data["trend_keys"]),
                "trend_count": factor_data["trend_count"],
                "weighted_mean": factor_data["weighted_mean"],  # DAILY return
                "weighted_variance": factor_data["weighted_variance"],
                "average_duration": factor_data["average_duration"],
                "total_duration": factor_data["total_duration"]
            })
    
    summary_df = pd.DataFrame(summary_data)
    
    print(f"\n✅ Impact metrics completed!")
    print(f"   Generated {len(summary_df)} factor metrics")
    print(f"   Market beta: {market_beta:.4f}")
    print(f"   Risk-free rate: {risk_free_rate:.1%}")
    
    return {
        "ticker": step3_result['ticker'],
        "market_beta": market_beta,
        "risk_free_rate": risk_free_rate,
        "aggregated_metrics": aggregated_metrics,
        "summary_df": summary_df,
        "data_period": step3_result['data_period']
    }

def summarise_factor_metrics(aggregated_metrics: Dict[str, Dict[str, Dict[str, Any]]]) -> pd.DataFrame:
    """
    Summarize factor metrics into DataFrame (exact same format as Quant Agent)
    
    Args:
        aggregated_metrics: Output from step4_impact_metrics
    
    Returns:
        pd.DataFrame: Summary of factor metrics
    """
    summary_data = []
    
    for scope, factors in aggregated_metrics.items():
        for factor_name, factor_data in factors.items():
            summary_data.append({
                "scope": scope,
                "factor": factor_name,
                "trend_keys": ", ".join(factor_data["trend_keys"]),
                "trend_count": factor_data["trend_count"],
                "weighted_mean": factor_data["weighted_mean"],  # DAILY return
                "weighted_variance": factor_data["weighted_variance"],
                "average_duration": factor_data["average_duration"],
                "total_duration": factor_data["total_duration"]
            })
    
    return pd.DataFrame(summary_data)

In [17]:
step4_result = step4_impact_metrics(step3_result)
summary_df = step4_result['summary_df']
summary_df

🔍 Step 4: Generating impact metrics for MSFT

 Processing 12 macro factors...
   ✅ Fed Rate Cut: μ=0.0036, σ²=0.0004
   ✅ Trump Tariff Implementation: μ=-0.0058, σ²=0.0006
   ✅ Trump Tariff Relief: μ=0.0051, σ²=0.0005
   ✅ Fed Policy Uncertainty: μ=0.0013, σ²=0.0001
   ✅ Fed Policy Clarity: μ=0.0032, σ²=0.0003
   ✅ Market Tech Valuation Concerns: μ=-0.0011, σ²=0.0001
   ✅ Market Tech Valuation Support: μ=0.0044, σ²=0.0004
   ✅ Geopolitical Tensions: μ=-0.0070, σ²=0.0007
   ✅ Inflation Concerns: μ=-0.0009, σ²=0.0001
   ✅ Stagflation Risk: μ=0.0005, σ²=0.0000
   ✅ Economic Growth Acceleration: μ=0.0037, σ²=0.0004
   ✅ Government Shutdown Threat: μ=0.0050, σ²=0.0005

 Processing 13 micro factors...
   ✅ Azure Growth Guidance Miss: μ=-0.0131, σ²=0.0020
   ✅ Azure Growth Guidance Beat: μ=0.0021, σ²=0.0003
   ✅ AI Revenue Acceleration: μ=0.0020, σ²=0.0003
   ✅ FTC Antitrust Investigation: μ=-0.0000, σ²=0.0000
   ✅ Regulatory Clearance: μ=-0.0002, σ²=0.0000
   ✅ Security Incident/Bug: μ=-0.00

,scope,factor,trend_keys,trend_count,weighted_mean,weighted_variance,average_duration,total_duration
0,macro,Fed Rate Cut,macro_fed_rate_cut,2,0.003559,0.000356,7.000000,14
1,macro,Trump Tariff Implementation,macro_trump_tariff_implementation,6,-0.005807,0.000581,9.166667,55
2,macro,Trump Tariff Relief,macro_trump_tariff_relief,5,0.005071,0.000507,20.200000,101
3,macro,Fed Policy Uncertainty,macro_fed_policy_uncertainty,4,0.001262,0.000126,10.000000,40
4,macro,Fed Policy Clarity,macro_fed_policy_clarity,5,0.003217,0.000322,13.600000,68
5,macro,Market Tech Valuation Concerns,macro_market_tech_valuation_concerns,5,-0.001093,0.000109,6.800000,34
6,macro,Market Tech Valuation Support,macro_market_tech_valuation_support,4,0.004358,0.000436,24.750000,99
7,macro,Geopolitical Tensions,macro_geopolitical_tensions,3,-0.006959,0.000696,7.666667,23
8,macro,Inflation Concerns,macro_inflation_concerns,4,-0.000860,0.000086,14.000000,56
9,macro,Stagflation Risk,macro_stagflation_risk,1,0.000492,0.000049,14.000000,14


### Summary Impaction Factor Metrics

In [18]:

def generate_impact_summary_schema(summary_df, language="English"):
    """
    Let LLM analyze the whole dataset and classify factors by row numbers
    """
    
    print("🔍 Letting LLM analyze dataset and classify factors...")
    
    # Add row index as column for LLM reference
    summary_df_with_index = summary_df.reset_index()
    
    # Create dataset string for LLM
    dataset_str = summary_df_with_index.to_string(index=False)
    
    # LLM analyzes and outputs mappings
    prompt = f"""
You are a financial analyst expert. Analyze the dataset below and classify factors into neutral categories.

DATASET:
{dataset_str}

REQUIREMENTS:
1. Classify into <=3 neutral categories for each scope (macro, micro, sector)
2. For each category, list the ROW NUMBERS (index column) that belong to it
3. Use neutral English names like "Monetary Policy", "Trade Policy", "Company Performance", etc.
4. Output clean JSON structure

5. If something is relative with the industry, the sector, (ex: mention about this industry valuation) consider it as sector 

6. You should extract the factors name and corresponding weight average in original numerical value 

EXAMPLE OUTPUT:
{{
    "macro_factors": [
        {{
            "factor_name": "Monetary Policy Impact",
            "row_numbers": [0, 8, 9],
            "max_return": "0.99",
            "min_return": "-0.57"
        }}
    ],
    "micro_factors": [
        {{
            "factor_name": "Company Performance", 
            "row_numbers": [20, 21],
            "max_return": "3.23",
            "min_return": "-1.28"
        }}
    ],
    "sector_factors": [
        {{
            "factor_name": "Technology Trends",
            "row_numbers": [13, 16],
            "max_return": "0.87",
            "min_return": "-0.34"
        }}
    ]
}}

Return ONLY the JSON structure, no other text.

You must return the text in the following language:
{language}
"""
    
    try:
        from LLM_Call_Agent import LLMCallAgent
        llm = LLMCallAgent()
        
        print("🤖 Calling LLM to analyze dataset...")
        response = llm.call_llm(prompt, model="deepseek-chat")
        
        # Parse JSON response
        json_start = response.find('{')
        json_end = response.rfind('}') + 1
        json_str = response[json_start:json_end]
        
        import json
        classification = json.loads(json_str)
        
        # Calculate max/min from actual weighted_mean values using row numbers
        result = {}
        
        for category, factors in classification.items():
            result[category] = []
            
            for factor_group in factors:
                row_numbers = factor_group['row_numbers']
                
                # Get weighted_mean values for these rows
                factor_df = summary_df.iloc[row_numbers]
                weighted_values = factor_df['weighted_mean']  # keep the original value
                
                updated_group = {
                    "factor_name": factor_group['factor_name'],
                    "row_numbers": row_numbers,
                    "sub_factors": list(factor_df['factor']),  # Factor names from those rows
                    "max_return": f"{weighted_values.max()}",
                    "min_return": f"{weighted_values.min()}",
                    "raw_values": list(weighted_values)  # Actual weighted_mean values
                }
                
                result[category].append(updated_group)
        
        print("✅ Schema Generated!")
        print(f"📊 Macro: {len(result.get('macro_factors', []))} groups")
        print(f"📊 Micro: {len(result.get('micro_factors', []))} groups") 
        print(f"📊 Sector: {len(result.get('sector_factors', []))} groups")

        return result
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return {"error": str(e)}


def convert_schema_to_compound_datasets(schema_result, summary_df):
    """
    Convert schema result into clean DataFrame with COMPOUND FORMULA applied
    Formula: (1 + weighted_mean)^average_duration - 1
    """
    import pandas as pd
    import numpy as np
    
    print("🔄 Converting schema to compound datasets...")
    
    # Safe formatting for arrays with < 3 elements
    def safe_format_array(arr, fmt, max_show=3):
        if len(arr) == 0:
            return '[]'
        elif len(arr) == 1:
            return f'[{fmt.format(arr[0])}]'
        elif len(arr) == 2:
            return f'[{fmt.format(arr[0])}, {fmt.format(arr[1])}]'
        else:
            shown = arr[:max_show]
            formatted = ', '.join([fmt.format(x) for x in shown])
            return f'[{formatted}, ...]' + (f', +{len(arr)-max_show} more' if len(arr) > max_show else '')
    
    # Convert to flat structure with compound calculations
    all_factors = []
    
    for category, factor_groups in schema_result.items():
        for group in factor_groups:
            category_name = category.replace('_factors', '').title()
            row_numbers = group['row_numbers']
            
            # Get the relevant rows from summary_df
            factor_rows = summary_df.iloc[row_numbers]
            
            # Calculate compound returns for each sub-factor
            weighted_means = factor_rows['weighted_mean'].values
            avg_durations = factor_rows['average_duration'].values
            
            # Apply compound formula: (1 + weighted_mean)^average_duration - 1
            compound_returns = []
            for wm, duration in zip(weighted_means, avg_durations):
                compound = (1 + wm) ** duration - 1
                compound_returns.append(compound)
            
            compound_returns = np.array(compound_returns)
            
            max_compound = compound_returns.max()
            min_compound = compound_returns.min()
            max_min_ratio = max_compound / min_compound if min_compound != 0 else float('inf')
            
            factor_row = {
                'category': category_name,
                'factor_name': group['factor_name'],
                'max_compound_return': max_compound,
                'min_compound_return': min_compound,
                'max_min_ratio': max_min_ratio,
                'return_range': max_compound - min_compound,
                'sub_factor_count': len(group['sub_factors']),
                'sub_factors': ' | '.join(group['sub_factors']),
                'row_numbers': str(group['row_numbers']),
                'weighted_means': safe_format_array(weighted_means, '{:.4f}'),
                'avg_durations': safe_format_array(avg_durations, '{:.1f}'),
                'compound_returns': safe_format_array(compound_returns, '{:.3f}')
            }
            
            all_factors.append(factor_row)
    
    # Create DataFrame
    clean_df = pd.DataFrame(all_factors)
    
    # Sort by max_compound_return (highest to lowest)
    clean_df = clean_df.sort_values('max_compound_return', ascending=False).reset_index(drop=True)
    
    # Display results
    print("📊 Compound Dataset Generated!")
    print(f"   Total factor categories: {len(clean_df)}")
    
    # Summary by category
    category_summary = clean_df.groupby('category').agg({
        'factor_name': 'count',
        'max_compound_return': ['max', 'min', 'mean'],
        'min_compound_return': ['max', 'min', 'mean'],
        'max_min_ratio': ['max', 'min', 'mean']
    }).round(4)
    
    print("\n📈 Summary by Category:")
    print(category_summary)
    
    # Show the clean dataset
    print("\n🎯 Compound Factor Dataset:")
    display_cols = ['category', 'factor_name', 'max_compound_return', 'min_compound_return', 'max_min_ratio', 'return_range', 'sub_factor_count']
    
    # Format numbers for display
    display_df = clean_df[display_cols].copy()
    for col in ['max_compound_return', 'min_compound_return', 'return_range']:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}")
    display_df['max_min_ratio'] = display_df['max_min_ratio'].apply(lambda x: f"{x:.2f}" if x != float('inf') else "inf")
    
    print(display_df.to_string(index=False))
    
    return clean_df


In [19]:
# USAGE - Copy these lines and run them:
schema_result = generate_impact_summary_schema(summary_df, language=language)
Factor_Risk_Reward = convert_schema_to_compound_datasets(schema_result, summary_df)



🔍 Letting LLM analyze dataset and classify factors...
🤖 Calling LLM to analyze dataset...
✅ Schema Generated!
📊 Macro: 3 groups
📊 Micro: 3 groups
📊 Sector: 1 groups
🔄 Converting schema to compound datasets...
📊 Compound Dataset Generated!
   Total factor categories: 7

📈 Summary by Category:
         factor_name max_compound_return                 min_compound_return  \
               count                 max     min    mean                 max   
category                                                                       
Macro              3              0.1076  0.0304  0.0609              0.0127   
Micro              3              0.0514 -0.0272  0.0080             -0.0066   
Sector             1              0.1136  0.1136  0.1136             -0.0249   

                         max_min_ratio                  
             min    mean           max     min    mean  
category                                                
Macro    -0.0520 -0.0171        3.5173 -2.5370 -0.3630 

In [20]:
Factor_Risk_Reward

,category,factor_name,max_compound_return,min_compound_return,max_min_ratio,return_range,sub_factor_count,sub_factors,row_numbers,weighted_means,avg_durations,compound_returns
0,Sector,Technology Sector Dynamics,0.113623,-0.024875,-4.567835,0.138497,5,Market Tech Valuation Concerns | Market Tech V...,"[5, 6, 17, 20, 21]","[-0.0011, 0.0044, -0.0050, ...], +2 more","[6.8, 24.8, 5.0, ...], +2 more","[-0.007, 0.114, -0.025, ...], +2 more"
1,Macro,Trade Policy,0.107587,-0.051990,-2.069374,0.159577,2,Trump Tariff Implementation | Trump Tariff Relief,"[1, 2]","[-0.0058, 0.0051]","[9.2, 20.2]","[-0.052, 0.108]"
2,Micro,Business Performance,0.051445,-0.063885,-0.805270,0.115330,5,Azure Growth Guidance Miss | Azure Growth Guid...,"[12, 13, 14, 18, 19]","[-0.0131, 0.0021, 0.0020, ...], +2 more","[5.0, 23.0, 21.7, ...], +2 more","[-0.064, 0.049, 0.045, ...], +2 more"
3,Macro,Monetary Policy,0.044648,0.012694,3.517307,0.031954,3,Fed Rate Cut | Fed Policy Uncertainty | Fed Po...,"[0, 3, 4]","[0.0036, 0.0013, 0.0032, ...]","[7.0, 10.0, 13.6, ...]","[0.025, 0.013, 0.045, ...]"
4,Macro,Economic Conditions,0.030392,-0.011980,-2.536986,0.042371,4,Inflation Concerns | Stagflation Risk | Econom...,"[8, 9, 10, 11]","[-0.0009, 0.0005, 0.0037, ...], +1 more","[14.0, 14.0, 2.0, ...], +1 more","[-0.012, 0.007, 0.007, ...], +1 more"
5,Micro,Regulatory Environment,-0.000292,-0.006609,0.044207,0.006317,3,FTC Antitrust Investigation | Regulatory Clear...,"[15, 16, 22]","[-0.0000, -0.0002, -0.0001, ...]","[16.3, 40.0, 12.2, ...]","[-0.000, -0.007, -0.002, ...]"
6,Micro,Market Position,-0.027232,-0.032825,0.829619,0.005593,2,Market Position Loss | Market Position Gain,"[23, 24]","[-0.0043, -0.0328]","[6.3, 1.0]","[-0.027, -0.033]"


### Event Risk Score

In [21]:
"""
Quant Impact Risk Analysis Pipeline
Input: Existing metrics DataFrame from previous analysis
Output: 1) Macro vs Micro Risk Share Index, 2) Separate Macro/Micro Volatility DataFrames, 3) Risk-Reward Ratio DataFrame, 4) Total Impact DataFrames
"""

import numpy as np
import pandas as pd
from typing import Dict, List, Any, Tuple

def calculate_trend_weighted_score(summary_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate trend-weighted scores for each factor
    
    Args:
        summary_df: DataFrame with columns ['scope', 'factor', 'trend_count', 'weighted_mean', 'weighted_variance']
    
    Returns:
        DataFrame with additional columns for trend-weighted metrics
    """
    # Calculate total trend count across all factors
    total_trends = summary_df['trend_count'].sum()
    
    # Calculate trend weight score (trend_count / total_trends)
    summary_df['trend_weight_score'] = summary_df['trend_count'] / total_trends
    
    # Calculate score-weighted mean and variance
    summary_df['score_weighted_mean'] = summary_df['trend_weight_score'] * summary_df['weighted_mean']
    summary_df['score_weighted_variance'] = summary_df['trend_weight_score'] * summary_df['weighted_variance']
    
    return summary_df

def calculate_macro_micro_risk_share(summary_df: pd.DataFrame) -> Dict[str, float]:
    """
    Calculate Macro vs Micro Risk Share Index
    
    Args:
        summary_df: DataFrame with trend-weighted metrics
    
    Returns:
        Dict with macro and micro risk share percentages
    """
    # Sum contributions by scope
    macro_contributions = summary_df[summary_df['scope'] == 'macro']['score_weighted_variance'].sum()
    micro_contributions = summary_df[summary_df['scope'] == 'micro']['score_weighted_variance'].sum()
    
    total_contributions = macro_contributions + micro_contributions
    
    if total_contributions == 0:
        return {"macro_risk_share": 0.0, "micro_risk_share": 0.0}
    
    macro_risk_share = (macro_contributions / total_contributions) * 100
    micro_risk_share = (micro_contributions / total_contributions) * 100
    
    return {
        "macro_risk_share": macro_risk_share,
        "micro_risk_share": micro_risk_share,
        "risk_environment": f"Current risk environment is {micro_risk_share:.1f}% company-driven, {macro_risk_share:.1f}% macro-driven."
    }

def calculate_factor_volatility_separated(summary_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Calculate Factor Volatility DataFrames with HIGH/LOW classification - SEPARATED by macro/micro
    
    Args:
        summary_df: DataFrame with trend-weighted metrics
    
    Returns:
        Tuple of (macro_volatility_df, micro_volatility_df)
    """
    volatility_df = summary_df.copy()
    
    # Calculate volatility (square root of weighted variance)
    volatility_df['volatility'] = np.sqrt(volatility_df['weighted_variance'])
    volatility_df['score_weighted_volatility'] = np.sqrt(volatility_df['score_weighted_variance'])
    
    # Separate macro and micro data
    macro_df = volatility_df[volatility_df['scope'] == 'macro'].copy()
    micro_df = volatility_df[volatility_df['scope'] == 'micro'].copy()
    
    # Calculate median thresholds SEPARATELY for macro and micro
    macro_median = macro_df['volatility'].median() if len(macro_df) > 0 else 0
    micro_median = micro_df['volatility'].median() if len(micro_df) > 0 else 0
    
    # Classify as HIGH or LOW volatility based on SEPARATE median thresholds
    macro_df['volatility_level'] = macro_df['volatility'].apply(
        lambda x: 'HIGH' if x >= macro_median else 'LOW'
    )
    micro_df['volatility_level'] = micro_df['volatility'].apply(
        lambda x: 'HIGH' if x >= micro_median else 'LOW'
    )
    
    # Sort by volatility (highest first) for each
    macro_df = macro_df.sort_values('volatility', ascending=False)
    micro_df = micro_df.sort_values('volatility', ascending=False)
    
    # Return only the required columns
    macro_volatility = macro_df[['scope', 'factor', 'volatility_level']].reset_index(drop=True)
    micro_volatility = micro_df[['scope', 'factor', 'volatility_level']].reset_index(drop=True)
    
    return macro_volatility, micro_volatility

def calculate_risk_reward_ratio(summary_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate Risk-Reward Ratio for each factor
    
    Args:
        summary_df: DataFrame with trend-weighted metrics
    
    Returns:
        DataFrame with risk-reward metrics (first 3 columns only)
    """
    risk_reward_df = summary_df.copy()
    
    # Calculate risk-reward ratio (Sharpe-style)
    # Avoid division by zero
    risk_reward_df['risk_reward_ratio'] = np.where(
        risk_reward_df['weighted_variance'] > 0,
        risk_reward_df['weighted_mean'] / np.sqrt(risk_reward_df['weighted_variance']),
        np.nan
    )
    
    # Sort by risk-reward ratio (highest first)
    risk_reward_df = risk_reward_df.sort_values('risk_reward_ratio', ascending=False, na_position='last')
    
    return risk_reward_df[['scope', 'factor', 'risk_reward_ratio']]

def calculate_final_impact_separated(summary_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Calculate Total Impact DataFrames - SEPARATED by macro/micro
    Formula: total_impact = (1 + weighted_mean)^average_duration - 1
    
    Args:
        summary_df: DataFrame with columns ['scope', 'factor', 'weighted_mean', 'average_duration']
    
    Returns:
        Tuple of (macro_total_impact_df, micro_total_impact_df)
    """
    # Separate macro and micro data
    macro_df = summary_df[summary_df['scope'] == 'macro'].copy()
    micro_df = summary_df[summary_df['scope'] == 'micro'].copy()
    
    # Calculate total impact using compound return formula
    # Formula: (1 + daily_return_rate)^days - 1
    macro_df['final_impact'] = (1 + macro_df['weighted_mean']) ** macro_df['average_duration'] - 1
    micro_df['final_impact'] = (1 + micro_df['weighted_mean']) ** micro_df['average_duration'] - 1
    
    # Sort by total impact (highest first)
    macro_df = macro_df.sort_values('final_impact', ascending=False)
    micro_df = micro_df.sort_values('final_impact', ascending=False)
    
    # Return only factor name and total impact
    macro_total_impact = macro_df[['factor', 'final_impact']].reset_index(drop=True)
    micro_total_impact = micro_df[['factor', 'final_impact']].reset_index(drop=True)
    
    return macro_total_impact, micro_total_impact

def quant_impact_risk_analysis(summary_df: pd.DataFrame) -> Tuple[Dict[str, float], pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Complete Quant Impact Risk Analysis Pipeline - UPDATED to return separate macro/micro volatility DataFrames and total impact DataFrames
    
    Args:
        summary_df: DataFrame with columns ['scope', 'factor', 'trend_count', 'weighted_mean', 'weighted_variance', 'total_duration']
    
    Returns:
        Tuple of:
        1. Macro vs Micro Risk Share Index
        2. Macro Factor Volatility DataFrame (with HIGH/LOW classification)
        3. Micro Factor Volatility DataFrame (with HIGH/LOW classification)  
        4. Risk-Reward Ratio DataFrame (first 3 columns only)
        5. Macro Total Impact DataFrame (factor, total_impact)
        6. Micro Total Impact DataFrame (factor, total_impact)
    """
    print("🔍 Starting Quant Impact Risk Analysis...")
    
    # Step 1: Calculate trend-weighted scores
    print("📊 Calculating trend-weighted scores...")
    enhanced_df = calculate_trend_weighted_score(summary_df.copy())
    
    # Step 2: Calculate Macro vs Micro Risk Share
    print("📈 Calculating Macro vs Micro Risk Share...")
    risk_share_index = calculate_macro_micro_risk_share(enhanced_df)
    
    # Step 3: Calculate Factor Volatility (SEPARATED by macro/micro)
    print("📉 Calculating Factor Volatility (separated)...")
    macro_volatility_df, micro_volatility_df = calculate_factor_volatility_separated(enhanced_df)
    
    # Step 4: Calculate Risk-Reward Ratio (first 3 columns only)
    print("⚖️ Calculating Risk-Reward Ratio...")
    risk_reward_df = calculate_risk_reward_ratio(enhanced_df)
    
    # Step 5: Calculate Total Impact (SEPARATED by macro/micro)
    print("💥 Calculating Total Impact (compound formula)...")
    macro_total_impact_df, micro_total_impact_df = calculate_final_impact_separated(enhanced_df)
    
    print("✅ Quant Impact Risk Analysis completed!")
    print(f"   📊 Macro volatility factors: {len(macro_volatility_df)}")
    print(f"   📊 Micro volatility factors: {len(micro_volatility_df)}")
    print(f"   💥 Macro total impact factors: {len(macro_total_impact_df)}")
    print(f"   💥 Micro total impact factors: {len(micro_total_impact_df)}")
    
    return risk_share_index, macro_volatility_df, micro_volatility_df, risk_reward_df, macro_total_impact_df, micro_total_impact_df

# =============================================================================
# USAGE EXAMPLE
# =============================================================================

In [22]:
risk_share_index, macro_volatility_df, micro_volatility_df, risk_reward_df, macro_total_impact_df, micro_total_impact_df = quant_impact_risk_analysis(summary_df)

🔍 Starting Quant Impact Risk Analysis...
📊 Calculating trend-weighted scores...
📈 Calculating Macro vs Micro Risk Share...
📉 Calculating Factor Volatility (separated)...
⚖️ Calculating Risk-Reward Ratio...
💥 Calculating Total Impact (compound formula)...
✅ Quant Impact Risk Analysis completed!
   📊 Macro volatility factors: 12
   📊 Micro volatility factors: 13
   💥 Macro total impact factors: 12
   💥 Micro total impact factors: 13


In [23]:
risk_share_index

{'macro_risk_share': np.float64(41.60995615535712),
 'micro_risk_share': np.float64(58.39004384464288),
 'risk_environment': 'Current risk environment is 58.4% company-driven, 41.6% macro-driven.'}

In [24]:
macro_volatility_df

,scope,factor,volatility_level
0,macro,Geopolitical Tensions,HIGH
1,macro,Trump Tariff Implementation,HIGH
2,macro,Trump Tariff Relief,HIGH
3,macro,Government Shutdown Threat,HIGH
4,macro,Market Tech Valuation Support,HIGH
5,macro,Economic Growth Acceleration,HIGH
6,macro,Fed Rate Cut,LOW
7,macro,Fed Policy Clarity,LOW
8,macro,Fed Policy Uncertainty,LOW
9,macro,Market Tech Valuation Concerns,LOW


In [25]:
micro_volatility_df

,scope,factor,volatility_level
0,micro,Market Position Gain,HIGH
1,micro,Azure Growth Guidance Miss,HIGH
2,micro,Cloud Business Disappointment,HIGH
3,micro,Security Incident/Bug,HIGH
4,micro,Market Position Loss,HIGH
5,micro,Analyst Upgrade,HIGH
6,micro,Cloud Business Exceedance,HIGH
7,micro,Azure Growth Guidance Beat,LOW
8,micro,AI Revenue Acceleration,LOW
9,micro,Regulatory Clearance,LOW


In [26]:
macro_total_impact_df

,factor,final_impact
0,Market Tech Valuation Support,0.113623
1,Trump Tariff Relief,0.107587
2,Fed Policy Clarity,0.044648
3,Government Shutdown Threat,0.030392
4,Fed Rate Cut,0.025177
5,Fed Policy Uncertainty,0.012694
6,Economic Growth Acceleration,0.007412
7,Stagflation Risk,0.006911
8,Market Tech Valuation Concerns,-0.007409
9,Inflation Concerns,-0.011980


In [27]:
micro_total_impact_df

,factor,final_impact
0,Cloud Business Exceedance,0.051445
1,Azure Growth Guidance Beat,0.048859
2,AI Revenue Acceleration,0.044564
3,FTC Antitrust Investigation,-0.000292
4,Analyst Downgrade,-0.001246
5,Legal Lawsuit Expansion,-0.001820
6,Regulatory Clearance,-0.006609
7,Analyst Upgrade,-0.018614
8,Security Incident/Bug,-0.024875
9,Market Position Loss,-0.027232


In [28]:
risk_reward_df

,scope,factor,risk_reward_ratio
2,macro,Trump Tariff Relief,0.225198
11,macro,Government Shutdown Threat,0.223659
6,macro,Market Tech Valuation Support,0.208751
10,macro,Economic Growth Acceleration,0.192330
0,macro,Fed Rate Cut,0.188641
4,macro,Fed Policy Clarity,0.179358
19,micro,Cloud Business Exceedance,0.127287
13,micro,Azure Growth Guidance Beat,0.117648
14,micro,AI Revenue Acceleration,0.115882
3,macro,Fed Policy Uncertainty,0.112347


### HTML 

In [29]:
# Import the function
from react_treemap_generator import generate_and_display_react_treemap

# Use with your existing variables
html_content = generate_and_display_react_treemap(
    macro_df=macro_total_impact_df,
    micro_df=micro_total_impact_df,
    Factor_Risk_Reward_dataset=Factor_Risk_Reward,
    risk_share_index=risk_share_index,
    language=language,
    ticker=ticker
)

Q&Q.AI treemap with Risk Analysis opened in browser! Language: English
📊 Ticker: MSFT
✅ Portfolio overview removed
✅ Multilingual support added
✅ Risk share summary integrated
